# 清除离群值 

随机孤立森林

In [ ]:
#生成用于处理利群数据的dataframe、并归一化
df_outlier = utils.Normalize_df(df_sta.reindex(columns = ['Dur','A_count','Count','Moment']))
print(len(df_outlier))
df_outlier.head()

1349


,Dur,A_count,Count,Moment
0,9.083546,1.666667,0.137805,9.575552
1,7.033248,0.000000,0.027358,8.556876
2,7.975277,6.666667,8.283514,4.011036
3,7.834612,0.833333,2.657817,3.938879
4,10.000000,0.000000,0.948424,5.021222


In [ ]:
myplot.Boxes([(df_outlier.Dur,"数据持续时间"),(df_outlier.Count,"数据量"),
              (df_outlier.A_count,"探针数量"),(df_outlier.Moment,"中位时刻")])

- 探针数量受被探测对象的移动范围影响
- 数据量受被探测对象的设备信号强度影响的同时，部分非探测对象设备也可能不间断发出数据，导致数据量激增
因此所经过的探针数量以及数据量不应当被视为噪声评估要素


在排除离群值时仅考虑数据持续时间和中位时刻

In [ ]:
from sklearn.ensemble import IsolationForest
rng = np.random.RandomState(42)
model_IF = IsolationForest(max_samples=256,random_state=rng,contamination='auto')
model_IF.fit(df_outlier[['Dur','Count']])
df_outlier['label'] = model_IF.predict(df_outlier[['Dur','Count']])
print(df_outlier.label.value_counts())

fig = px.scatter(df_outlier,y = 'Count',x = 'Dur',color='label')
fig.update_traces(marker_size = 5)
fig.update_layout(scattermode='group')
fig.show()

label
 1    1068
-1     281
Name: count, dtype: int64


In [ ]:
df_cluster = df_outlier[df_outlier.label == 1].reset_index().drop('index',axis = 1)
df_cluster = utils.Normalize_df(df_cluster,cols = ['Dur','Moment','Count'])
myplot.Scatter_3D(df_cluster,'Dur','Count','Moment','label',color_name='A_count')